In [ ]:
# ==========================================================
#  CAU HINH - CHINH CAC THAM SO O DAY TRUOC KHI CHAY
# ==========================================================
MODEL_ID            = "xiaomi-research/Xiaomi-OCR-0"

PIN_TRANSFORMERS    = "5.17.0"

# --- Che do huan luyen ---
# Dat WARM_START_ADAPTER = None de train tu dau (fresh start)
# Dat duong dan adapter neu muon warm-start tu checkpoint co san
WARM_START_ADAPTER  = "/tmp/dataset/lora_adapter_v9"

# --- Sieu tham so ---
NUM_EPOCHS     = 5
LR             = 3e-5
LR_FP16_CAP    = 3e-5
LR_SCHEDULER   = "cosine"
LORA_R         = 32
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
INCLUDE_VISION = True
TARGET_EFF_BS  = 64
WARMUP_STEPS   = 30
WEIGHT_DECAY   = 0.01
USE_MULTI_GPU  = False

TOK_BUDGET_15GB  = 1536
AUTO_SMOKE_FIRST = False
TIME_LIMIT_H     = 8.5
RESUME           = False
SHORT_PROMPT     = "Transcribe the handwritten text in this cell. Output only the text."
print("Config san sang!")


In [ ]:
import os, sys, subprocess, shutil, json, math, re
from pathlib import Path

try:
    _smi = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
except FileNotFoundError:
    _smi = ''
print(_smi)
N_GPU_TOTAL = len([l for l in _smi.splitlines() if l.startswith('GPU')])
if N_GPU_TOTAL == 0:
    raise RuntimeError('Khong thay GPU. Settings -> Accelerator -> chon GPU P100 hoac T4, roi chay lai.')
N_GPU = N_GPU_TOTAL if USE_MULTI_GPU else 1
print(f'Phat hien {N_GPU_TOTAL} GPU, se dung {N_GPU}')

import torch
HAS_BF16   = torch.cuda.get_device_capability(0)[0] >= 8
EFFECTIVE_LR = LR if HAS_BF16 else min(LR, LR_FP16_CAP)
print(f'Precision: {"bf16" if HAS_BF16 else "fp16"} | LR: {EFFECTIVE_LR}')

WORK       = Path('/kaggle/working')
OUTPUT_DIR = WORK / 'ocr_finetune'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER      = OUTPUT_DIR / 'lora_adapter'
ADAPTER_FILE = ADAPTER / 'adapter_model.safetensors'
print('Output dir:', OUTPUT_DIR)


In [ ]:
def sh(args, env=None):
    r = subprocess.run(args, capture_output=True, text=True, env=env)
    return r.returncode, r.stdout + r.stderr

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)

tf_spec = f'transformers=={PIN_TRANSFORMERS}' if PIN_TRANSFORMERS else 'transformers'
rc, out = sh([sys.executable, '-m', 'pip', 'install', '-q', tf_spec,
              'peft>=0.10', 'accelerate>=0.28', 'huggingface_hub',
              'einops', 'sentencepiece', 'pillow'])
if rc == 0:
    print('pip install: OK')
else:
    print('pip install: LOI (van thu tiep)')
    print(out[-1000:])


In [ ]:
import zipfile, shutil
from pathlib import Path

LOCAL_DATA = Path("/tmp/dataset")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

print("Dang chuan bi du lieu tai /tmp/dataset...")
for j in Path("/kaggle/input").rglob("*.jsonl"):
    shutil.copy(j, LOCAL_DATA / j.name)
    print(f"Copied {j.name}")

for item in Path("/kaggle/input").rglob("*"):
    if item.is_file() and item.suffix == ".zip":
        stem = item.stem
        target = LOCAL_DATA / stem
        if not target.exists() or not any(target.iterdir()):
            print(f"Unpacking {item.name} -> {target}...")
            shutil.unpack_archive(item, target)
    elif item.is_dir() and item.name in ["train", "eval_720", "lora_adapter_v1",
                                          "lora_adapter_v9", "lora_adapter"]:
        target = LOCAL_DATA / item.name
        if not target.exists():
            print(f"Copying folder {item.name} -> {target}...")
            shutil.copytree(item, target, dirs_exist_ok=True)

DATASET_DIR = LOCAL_DATA
print("Du lieu san sang! Cau truc:")
for p in LOCAL_DATA.iterdir():
    if p.is_dir():
        print(f"  {p.name}/ ({len(list(p.rglob('*')))} files)")
    else:
        print(f"  {p.name} ({p.stat().st_size} bytes)")


In [ ]:
%%writefile /kaggle/working/ocr_common.py
import os, sys, json
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset

SEARCH_ROOT = Path(os.environ.get('CC_SEARCH_ROOT', '/kaggle/input'))
IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}


def prepare_env():
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


def patch_peft_torchao():
    for _n, _m in list(sys.modules.items()):
        if _n.startswith('peft') and hasattr(_m, 'is_torchao_available'):
            _m.is_torchao_available = lambda: False


def find_file(name, root):
    root = Path(root)
    if (root / name).exists():
        return root / name
    hits = list(root.rglob(name)) if root.exists() else []
    if hits:
        return hits[0]
    hits = list(SEARCH_ROOT.rglob(name)) if SEARCH_ROOT.exists() else []
    if hits:
        return hits[0]
    raise FileNotFoundError(f'Khong tim thay {name} trong {root}')


def _content_text(content):
    if isinstance(content, list):
        return ''.join(p.get('text', '') for p in content if isinstance(p, dict))
    return content


def get_label(rec):
    lab = rec.get('label')
    if lab is not None:
        return lab if isinstance(lab, str) else json.dumps(lab, ensure_ascii=False)
    return _content_text(rec['messages'][-1]['content'])


def assistant_text(rec):
    return _content_text(rec['messages'][-1]['content'])


def clean_messages(msgs):
    out = []
    for turn in msgs:
        if turn['role'] == 'user' and isinstance(turn['content'], list):
            content = [{'type': 'image'} if p.get('type') == 'image' else p for p in turn['content']]
            out.append({'role': turn['role'], 'content': content})
        else:
            out.append(turn)
    return out


class OCRDataset(Dataset):
    def __init__(self, jsonl_path, root, prompt_override=None):
        self.root = Path(root)
        p = find_file(Path(jsonl_path).name, self.root)
        self.jsonl_path = p
        self.jsonl_dir = p.parent
        with open(p, encoding='utf-8') as f:
            self.records = [json.loads(l) for l in f if l.strip()]
        if prompt_override:
            for rec in self.records:
                for turn in rec['messages']:
                    if turn['role'] == 'user' and isinstance(turn['content'], list):
                        for part in turn['content']:
                            if part.get('type') == 'text':
                                part['text'] = prompt_override
        self._index = None
        first = self._resolve(self.records[0]['image'])
        print(f'[Dataset] {len(self.records)} records <- {p}  (anh mau: {first})', flush=True)

    def _resolve(self, rel):
        for base in (self.jsonl_dir, self.root, self.jsonl_dir.parent):
            c = base / rel
            if c.exists():
                return c
        if self._index is None:
            self._index = {}
            for base in {self.root, self.jsonl_dir}:
                for f in Path(base).rglob('*'):
                    if f.is_file() and f.suffix.lower() in IMG_EXT:
                        self._index.setdefault(f.name, f)
        name = Path(rel).name
        if name in self._index:
            return self._index[name]
        raise FileNotFoundError(f'Khong thay anh: {rel}')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = Image.open(self._resolve(rec['image'])).convert('RGB')
        return {'image': img, 'label': get_label(rec), 'messages': rec['messages']}

    def resolve(self, rel):
        return self._resolve(rel)


class OCRDataCollator:
    """Chi tinh loss tren phan tra loi cua assistant.
    Uu tien: sau the </think> cuoi cung. Du phong: sau '<|im_start|>assistant'."""

    def __init__(self, processor, max_length=1024):
        self.processor = processor
        self.max_length = max_length
        self.tok = processor.tokenizer
        unk = self.tok.unk_token_id
        def _id(t):
            i = self.tok.convert_tokens_to_ids(t)
            return None if (i is None or (unk is not None and i == unk)) else i
        self.im_end_id = _id('<|im_end|>')
        self.im_start_id = _id('<|im_start|>')
        self.think_end_id = _id('</think>')
        self._warned = False
        self.n_empty = 0

    def _find_span(self, l):
        ends = [j for j, t in enumerate(l) if t == self.im_end_id] if self.im_end_id is not None else []
        if not ends:
            return None
        start = None
        if self.think_end_id is not None:
            think = [j for j, t in enumerate(l) if t == self.think_end_id]
            if think:
                start = think[-1] + 1
        if start is None and self.im_start_id is not None:
            st = [j for j, t in enumerate(l) if t == self.im_start_id]
            if st:
                start = st[-1] + 1
                # bo qua 'assistant' va xuong dong
                while start < len(l) and l[start] != self.im_end_id and \
                        self.tok.decode([l[start]]).strip() in ('', 'assistant'):
                    start += 1
        if start is None:
            return None
        while start < len(l) and l[start] != self.im_end_id and self.tok.decode([l[start]]).strip() == '':
            start += 1
        end = ends[-1]
        return (start, end) if start <= end else None

    def __call__(self, batch):
        images = [b['image'] for b in batch]
        texts = [self.processor.apply_chat_template(clean_messages(b['messages']),
                                                    tokenize=False, add_generation_prompt=False)
                 for b in batch]
        enc = self.processor(text=texts, images=images, return_tensors='pt', padding=True)
        if enc['input_ids'].shape[1] > self.max_length and not self._warned:
            print(f"Chuoi dai {enc['input_ids'].shape[1]} > {self.max_length} token", flush=True)
            self._warned = True
        ids_all = enc['input_ids']
        labels = torch.full_like(ids_all, -100)
        for i, ids in enumerate(ids_all):
            span = self._find_span(ids.tolist())
            if span is None:
                self.n_empty += 1
                continue
            s, e = span
            labels[i, s:e + 1] = ids[s:e + 1]
        enc['labels'] = labels
        return enc

In [ ]:
%%writefile /kaggle/working/train_ocr.py
import os, sys, json, math, time
sys.path.insert(0, os.environ.get('CC_WORK', '/kaggle/working'))
import ocr_common as cc
cc.prepare_env()

import torch
from pathlib import Path
from torch.utils.data import Subset
from transformers import (AutoProcessor, AutoModelForImageTextToText,
                          TrainingArguments, Trainer,
                          TrainerCallback, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, PeftModel
from transformers.trainer_utils import get_last_checkpoint
cc.patch_peft_torchao()

CFG        = json.load(open(os.environ['CFG_PATH']))
rank       = int(os.environ.get('RANK', 0))
local_rank = int(os.environ.get('LOCAL_RANK', 0))
world      = int(os.environ.get('WORLD_SIZE', 1))
is_main    = (rank == 0)
torch.cuda.set_device(local_rank)
USE_BF16   = torch.cuda.get_device_capability(local_rank)[0] >= 8
DTYPE      = torch.bfloat16 if USE_BF16 else torch.float16
T0         = time.time()


def log(*a):
    if is_main:
        print(*a, flush=True)


DATASET_DIR = Path(CFG['dataset_dir'])
OUTPUT_DIR  = Path(CFG['output_dir'])
log(f'world={world} | bs={CFG["per_device_bs"]} | accum={CFG["accum"]} | '
    f'eff={CFG["per_device_bs"]*CFG["accum"]*world} | {"bf16" if USE_BF16 else "fp16"} | lr={CFG["lr"]}')

processor = AutoProcessor.from_pretrained(CFG['model_id'], trust_remote_code=True)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

try:
    model = AutoModelForImageTextToText.from_pretrained(
        CFG['model_id'], dtype=DTYPE, device_map={'': local_rank}, trust_remote_code=True)
except TypeError:
    model = AutoModelForImageTextToText.from_pretrained(
        CFG['model_id'], torch_dtype=DTYPE, device_map={'': local_rank}, trust_remote_code=True)
model.config.use_cache = False

LANG_KEYS = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
targets, n_lang, n_vis = [], 0, 0
for n, m in model.named_modules():
    if not isinstance(m, torch.nn.Linear):
        continue
    is_vis = ('visual' in n) or ('vision' in n)
    if is_vis:
        if CFG['include_vision']:
            targets.append(n); n_vis += 1
    elif n.split('.')[-1] in LANG_KEYS:
        targets.append(n); n_lang += 1
log(f'LoRA: {n_lang} lang + {n_vis} vision  (r={CFG["lora_r"]}, alpha={CFG["lora_alpha"]})')
assert targets, 'Khong tim thay lop Linear nao de gan LoRA'
warm_start_path = CFG.get('warm_start_adapter')
if warm_start_path and Path(warm_start_path).exists():
    log(f"==> [WARM-START] Nạp trọng số LoRA từ V1: {warm_start_path}")
    model = PeftModel.from_pretrained(model, str(warm_start_path), is_trainable=True)
else:
    log("==> Khởi tạo LoRA mới từ đầu")
    model = get_peft_model(model, LoraConfig(
        r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
        target_modules=targets,
        lora_dropout=CFG.get('lora_dropout', 0.05),
        bias='none'))
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
if is_main:
    model.print_trainable_parameters()

prompt   = CFG.get('short_prompt')
train_ds = cc.OCRDataset(DATASET_DIR / 'train.jsonl', DATASET_DIR, prompt_override=prompt)
val_ds   = cc.OCRDataset(DATASET_DIR / 'val.jsonl',   DATASET_DIR, prompt_override=prompt)
eval_ds  = Subset(val_ds, list(range(min(400, len(val_ds)))))
SMOKE    = CFG.get('smoke', False)
if SMOKE:
    train_ds.records = train_ds.records[:96]
    eval_ds = Subset(val_ds, list(range(min(16, len(val_ds)))))
    log('SMOKE TEST: 96 mau, 6 step')
collator = cc.OCRDataCollator(processor)


class TimeLimit(TrainerCallback):
    def __init__(self, hours):
        self.limit = hours * 3600
        self.t0 = None
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > self.limit:
            log('Het gio -> luu adapter va dung.')
            control.should_training_stop = True
        return control


class Heartbeat(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        msg = f'[{time.time()-T0:.0f}s] step {state.global_step}/{state.max_steps}'
        if logs:
            if 'loss' in logs:
                msg += f'  loss={logs["loss"]:.4f}'
            if 'eval_loss' in logs:
                msg += f'  eval_loss={logs["eval_loss"]:.4f}'
        log(msg)


class NaNGuard(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            for k in ('loss', 'eval_loss'):
                if k in logs and not math.isfinite(float(logs[k])):
                    raise RuntimeError(f'{k} = NaN/Inf -> dung. Giam LR (vd 5e-5) hoac dung GPU ho tro bf16.')


bs, accum = CFG['per_device_bs'], CFG['accum']
steps_per_epoch = math.ceil(len(train_ds) / (bs * accum * world))
every = 3 if SMOKE else max(25, steps_per_epoch // 3)
log(f'steps_per_epoch={steps_per_epoch} | eval+save every={every} steps')

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=CFG['epochs'],
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=accum,
    per_device_eval_batch_size=bs,
    prediction_loss_only=True,
    learning_rate=CFG['lr'],
    lr_scheduler_type=CFG.get('lr_scheduler', 'cosine'),
    warmup_steps=(2 if SMOKE else CFG.get('warmup_steps', 50)),
    weight_decay=CFG.get('weight_decay', 0.01),
    max_grad_norm=1.0,
    fp16=(not USE_BF16), bf16=USE_BF16,
    gradient_checkpointing=False,
    optim='adamw_torch',
    max_steps=(6 if SMOKE else -1),
    logging_steps=(1 if SMOKE else 10),
    logging_first_step=True,
    eval_strategy='steps', eval_steps=every,
    save_strategy='steps', save_steps=every,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    disable_tqdm=True,
    remove_unused_columns=False,
    dataloader_num_workers=CFG.get('workers', 2),
    dataloader_pin_memory=True,
    ddp_find_unused_parameters=False,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator,
    callbacks=[TimeLimit(CFG.get('time_limit_h', 8.5)), Heartbeat(), NaNGuard(),
               EarlyStoppingCallback(early_stopping_patience=4)],
)

resume_ckpt = None
if CFG.get('resume', False):
    resume_ckpt = get_last_checkpoint(str(OUTPUT_DIR))
    if resume_ckpt:
        log(f'Resume tu: {resume_ckpt}')

trainer.train(resume_from_checkpoint=resume_ckpt)

if collator.n_empty:
    log(f'CANH BAO: {collator.n_empty} mau khong tim thay vung nhan (loss mask rong).')

adapter_path = OUTPUT_DIR / 'lora_adapter'
if is_main:
    trainer.model.save_pretrained(str(adapter_path))
    processor.save_pretrained(str(adapter_path))
log(f'Adapter da luu tai: {adapter_path}')
log(f'Tong thoi gian: {(time.time()-T0)/60:.1f} phut')


In [ ]:
%%writefile /kaggle/working/eval_ocr.py
import os, sys, json, traceback
sys.path.insert(0, os.environ.get('CC_WORK', '/kaggle/working'))
import ocr_common as cc
cc.prepare_env()

import torch
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
cc.patch_peft_torchao()

CFG     = json.load(open(os.environ['CFG_PATH']))
N_REQ   = int(os.environ.get('EVAL_N', '100'))
OUT     = os.environ.get('EVAL_OUT')
DTYPE   = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
ADAPTER = Path(CFG['output_dir']) / 'lora_adapter'


def cer(gt, pred):
    if not gt:
        return 0.0 if not pred else 1.0
    m, n = len(gt), len(pred)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if gt[i-1] == pred[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n] / m


def main():
    processor = AutoProcessor.from_pretrained(CFG['model_id'], trust_remote_code=True)
    val_ds = cc.OCRDataset(Path(CFG['dataset_dir']) / 'val.jsonl', CFG['dataset_dir'],
                           prompt_override=CFG.get('short_prompt'))
    try:
        base = AutoModelForImageTextToText.from_pretrained(
            CFG['model_id'], dtype=DTYPE, device_map={'': 0}, trust_remote_code=True)
    except TypeError:
        base = AutoModelForImageTextToText.from_pretrained(
            CFG['model_id'], torch_dtype=DTYPE, device_map={'': 0}, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, str(ADAPTER)).eval()
    max_new = int(CFG.get('max_new', 24))

    def predict_one(rec):
        image = Image.open(val_ds.resolve(rec['image'])).convert('RGB')
        try:
            text = processor.apply_chat_template(
                cc.clean_messages(rec['messages'][:-1]), tokenize=False,
                add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            text = processor.apply_chat_template(
                cc.clean_messages(rec['messages'][:-1]), tokenize=False,
                add_generation_prompt=True)
        inp = processor(text=text, images=[image], return_tensors='pt')
        inp = {k: (v.to(model.device, dtype=DTYPE) if v.is_floating_point() else v.to(model.device))
               for k, v in inp.items()}
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new, do_sample=False, use_cache=True)
        return processor.tokenizer.decode(
            out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    N = min(N_REQ, len(val_ds.records))
    n_exact = 0; total_cer = 0.0; rows = []
    for i, rec in enumerate(val_ds.records[:N]):
        gt   = cc.assistant_text(rec)
        pred = predict_one(rec)
        ok   = (pred.strip() == gt.strip())
        c    = cer(gt, pred)
        n_exact += ok; total_cer += c
        rows.append({'image': rec['image'], 'gt': gt, 'pred': pred, 'ok': ok, 'cer': round(c, 3)})
        if (i + 1) % 50 == 0:
            print(f'  [{i+1}/{N}] exact={n_exact/(i+1)*100:.1f}%  CER={total_cer/(i+1)*100:.1f}%', flush=True)

    print('\n' + '=' * 56)
    print(f'Exact match: {n_exact/N*100:.2f}%  ({N} o val)')
    print(f'Average CER: {total_cer/N*100:.2f}%')
    wrong = [r for r in rows if not r['ok']]
    print(f'So o sai: {len(wrong)}/{N}')
    for r in wrong[:8]:
        print(f'  GT: {r["gt"][:60]!r}  PRED: {r["pred"][:60]!r}  CER={r["cer"]:.2f}')
    print('=' * 56)
    if OUT:
        json.dump(rows, open(OUT, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
        print(f'Saved -> {OUT}')

    # === ĐÁNH GIÁ TẬP 720 Ô M/D THỰC TẾ TRÊN GPU ===
    eval_720_path = Path("/tmp/dataset/eval_720/eval_720.jsonl")
    if eval_720_path.exists():
        print("\n" + "=" * 60)
        print("🚀 BẮT ĐẦU ĐÁNH GIÁ 720 Ô M/D TRÊN GPU...")
        print("=" * 60)
        processor.tokenizer.padding_side = 'right'  # TUYỆT ĐỐI không padding ở đầu
        
        with open(eval_720_path, encoding='utf-8') as f:
            recs_720 = [json.loads(line) for line in f if line.strip()]
        
        n_720 = len(recs_720)
        ok_720 = 0
        cer_720 = 0.0
        rows_720 = []
        
        for idx, r in enumerate(recs_720):
            sid = r["id"]
            gt = r["gt"]
            img_f = Path("/tmp/dataset") / r["image"]
            try:
                im = Image.open(img_f).convert("RGB")
                eval_prompt = CFG.get("short_prompt", "Transcribe the handwritten text in this cell. Output only the text.")
                text_prompt = processor.apply_chat_template(
                    [{"role": "user", "content": [
                        {"type": "image"},
                        {"type": "text", "text": eval_prompt}
                    ]}],
                    tokenize=False, add_generation_prompt=True
                )
                inp = processor(text=text_prompt, images=[im], return_tensors="pt")
                inp = {k: (v.to(model.device, dtype=DTYPE) if v.is_floating_point() else v.to(model.device))
                       for k, v in inp.items()}
                with torch.no_grad():
                    out = model.generate(**inp, max_new_tokens=24, do_sample=False, use_cache=True)
                pred = processor.tokenizer.decode(
                    out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True
                ).strip()
            except Exception as e:
                pred = f"__ERR__: {e}"
            
            is_match = (pred.strip() == gt.strip())
            c_dist = cer(gt, pred)
            ok_720 += is_match
            cer_720 += c_dist
            rows_720.append({
                "id": sid,
                "image": r["image"],
                "gt": gt,
                "pred": pred,
                "exact_match": is_match,
                "cer": round(c_dist, 4)
            })
            
            if (idx + 1) % 100 == 0 or (idx + 1) == n_720:
                print(f"  [{idx+1}/{n_720}] Exact: {ok_720/(idx+1)*100:.2f}% | CER: {cer_720/(idx+1)*100:.2f}%", flush=True)

        em_pct_720 = ok_720 / n_720 * 100
        avg_cer_720 = cer_720 / n_720 * 100
        print("\n" + "=" * 60)
        print(f"🎉 KẾT QUẢ 720 Ô M/D: {ok_720}/{n_720} ({em_pct_720:.2f}%)")
        print(f"   CER trung bình: {avg_cer_720:.2f}%")
        print("=" * 60)
        
        pred_out_file = Path(CFG["output_dir"]) / "predictions_720.json"
        with open(pred_out_file, "w", encoding="utf-8") as f:
            json.dump(rows_720, f, ensure_ascii=False, indent=2)
            
        sum_out_file = Path(CFG["output_dir"]) / "summary_720.json"
        with open(sum_out_file, "w", encoding="utf-8") as f:
            json.dump({
                "total_samples": n_720,
                "exact_match_count": ok_720,
                "exact_match_pct": em_pct_720,
                "avg_cer_pct": avg_cer_720,
                "failures_count": n_720 - ok_720
            }, f, indent=2)
        print(f"Đã lưu kết quả đánh giá 720 ô -> {pred_out_file}")


try:
    main()
except Exception:
    traceback.print_exc()
    sys.exit(1)


In [ ]:
import sys, random, collections
sys.path.insert(0, str(WORK))
import importlib, ocr_common as cc
importlib.reload(cc)
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

train_ds = cc.OCRDataset(DATASET_DIR / 'train.jsonl', DATASET_DIR, prompt_override=SHORT_PROMPT)
val_ds   = cc.OCRDataset(DATASET_DIR / 'val.jsonl',   DATASET_DIR, prompt_override=SHORT_PROMPT)
col      = cc.OCRDataCollator(processor)
print(f'think_end_id={col.think_end_id} | im_start_id={col.im_start_id} | im_end_id={col.im_end_id}')
assert col.im_end_id is not None, 'Tokenizer không có <|im_end|> — cần chỉnh collator theo chat template của model.'

# 1) Phân bố nhãn
cnt = collections.Counter(cc.get_label(r) for r in train_ds.records)
print(f'\nSố nhãn khác nhau: {len(cnt)}')
for k, v in cnt.most_common(10):
    print(f'  {v:5d} ({v/len(train_ds)*100:.1f}%)  {k[:80]}')

# 2) Kiểm tra loss mask: vùng tính loss phải đúng bằng nhãn
random.seed(42)
idx = random.sample(range(len(train_ds)), min(64, len(train_ds)))
lens, mismatch, empty = [], 0, 0
print('\n--- Kiểm tra loss mask (3 mẫu đầu) ---')
for k, i in enumerate(idx):
    item = train_ds[i]
    b = col([item])
    lens.append(b['input_ids'].shape[1])
    lab = b['labels'][0]
    n_sup = int((lab != -100).sum())
    if n_sup == 0:
        empty += 1
        continue
    loss_txt = processor.tokenizer.decode(lab[lab != -100]).replace('<|im_end|>', '').strip()
    if loss_txt != item['label'].strip():
        mismatch += 1
    if k < 3:
        print(f'  loss tokens = {loss_txt!r:50s} | GT = {item["label"][:60]!r}')
print(f'Mẫu mask rỗng: {empty}/{len(idx)} | mask không khớp nhãn: {mismatch}/{len(idx)}')
assert empty == 0, 'Có mẫu không có token nào để tính loss. Xem lại chat template / collator trước khi train.'
if mismatch:
    print('CẢNH BÁO: vùng loss khác nhãn gốc ở một số mẫu (có thể do token đặc biệt). Hãy xem 3 mẫu in ở trên.')
print(f'\nĐộ dài chuỗi: trung bình {sum(lens)/len(lens):.0f}, lớn nhất {max(lens)}')

# 3) Chọn batch size theo VRAM thực tế
q = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                   capture_output=True, text=True).stdout.strip().splitlines()
gpu_name, mem_mb = [x.strip() for x in q[0].split(',')]
mem_gb = float(mem_mb) / 1024
TOK_BUDGET = int(TOK_BUDGET_15GB * mem_gb / 15)
PER_DEV_BS = max(1, min(16, TOK_BUDGET // max(lens)))
print(f'\nGPU: {gpu_name} ({mem_gb:.1f} GB) | token budget {TOK_BUDGET} -> per-device batch = {PER_DEV_BS}')

label_lens = [len(processor.tokenizer(cc.assistant_text(r))['input_ids']) for r in train_ds.records[:200]]
MAX_NEW = min(32, max(label_lens) + 4)
print(f'max_new_tokens khi eval: {MAX_NEW}')

# 4) Kiểm tra ảnh
missing = 0
for r in random.sample(train_ds.records, min(200, len(train_ds.records))):
    try:
        train_ds.resolve(r['image'])
    except FileNotFoundError:
        missing += 1
print(f'\nKiểm tra 200 ảnh ngẫu nhiên: {missing} thiếu', 'OK' if missing == 0 else 'LỖI')
assert missing == 0, 'Thiếu ảnh — kiểm tra lại cấu trúc dataset.'

In [ ]:
SCRIPT      = WORK / 'train_ocr.py'
EVAL_SCRIPT = WORK / 'eval_ocr.py'
CFG_PATH    = WORK / 'cfg.json'
STEP_RE     = re.compile(r'\] step \d+/\d+')

def clean_output():
    for d in OUTPUT_DIR.glob('checkpoint-*'):
        shutil.rmtree(d, ignore_errors=True)
    if ADAPTER.exists():
        shutil.rmtree(ADAPTER)

def base_env():
    env = os.environ.copy()
    env.update(CFG_PATH=str(CFG_PATH), CC_WORK=str(WORK), CC_SEARCH_ROOT=str(LOCAL_DATA),
               OMP_NUM_THREADS='1', PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false',
               PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
               NCCL_P2P_DISABLE='1', NCCL_IB_DISABLE='1')
    return env

def run_stream(cmd, env):
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    started, tail = False, []
    for line in p.stdout:
        print(line, end='', flush=True)
        if STEP_RE.search(line):
            started = True
        tail.append(line); tail = tail[-400:]
    return p.wait(), started, ''.join(tail)

def write_cfg(n_gpu, bs, smoke, resume):
    cfg = dict(
        model_id=MODEL_ID, dataset_dir=str(DATASET_DIR), output_dir=str(OUTPUT_DIR),
        epochs=NUM_EPOCHS, lr=EFFECTIVE_LR, lr_scheduler=LR_SCHEDULER,
        lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        include_vision=INCLUDE_VISION, warmup_steps=WARMUP_STEPS, weight_decay=WEIGHT_DECAY,
        time_limit_h=TIME_LIMIT_H, short_prompt=SHORT_PROMPT, smoke=smoke, resume=resume,
        warm_start_adapter=WARM_START_ADAPTER,
        max_new=MAX_NEW, per_device_bs=bs,
        accum=max(1, math.ceil(TARGET_EFF_BS / (bs * n_gpu))),
        workers=max(2, min(4, (os.cpu_count() or 2))) if n_gpu == 1 else 2)
    json.dump(cfg, open(CFG_PATH, 'w'))
    return cfg

def train_once(n_gpu, bs, smoke=False, resume=False):
    cfg = write_cfg(n_gpu, bs, smoke, resume)
    env = base_env()
    if n_gpu > 1:
        cmd = [sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node', str(n_gpu),
               '--master_port', '29517', str(SCRIPT)]
    else:
        env['CUDA_VISIBLE_DEVICES'] = '0'
        cmd = [sys.executable, str(SCRIPT)]
    print(f'\n{"[SMOKE] " if smoke else ""}Train | {n_gpu} GPU | bs={bs} | accum={cfg["accum"]} '
          f'| eff_bs={bs*cfg["accum"]*n_gpu} | lr={cfg["lr"]}\n', flush=True)
    return run_stream(cmd, env)

def evaluate(n, out_name):
    write_cfg(1, PER_DEV_BS, False, False)
    env = base_env()
    env.update(EVAL_N=str(n), EVAL_OUT=str(OUTPUT_DIR / out_name), CUDA_VISIBLE_DEVICES='0')
    print(f'\nĐánh giá {n} ô...\n', flush=True)
    return run_stream([sys.executable, str(EVAL_SCRIPT)], env)[0]

def run_phase(smoke, resume=False):
    """Chạy train với cơ chế hạ batch khi OOM. Trả về True nếu có adapter."""
    bs_list = [PER_DEV_BS]
    while bs_list[-1] > 1 and len(bs_list) < 4:
        bs_list.append(max(1, bs_list[-1] // 2))
    n_gpu, i = N_GPU, 0
    while i < len(bs_list):
        rc, started, tail = train_once(n_gpu, bs_list[i], smoke=smoke, resume=resume)
        if ADAPTER_FILE.exists():
            return True
        oom = ('OutOfMemoryError' in tail) or ('out of memory' in tail.lower())
        if 'NaN/Inf' in tail:
            print('\nLoss bị NaN/Inf. Giảm LR (LR_FP16_CAP / LR) hoặc dùng GPU có bf16.')
            return False
        if n_gpu > 1 and not started and not oom:
            print('\nMulti-GPU không khởi động -> thử 1 GPU\n', flush=True)
            n_gpu = 1
            continue
        if oom and i + 1 < len(bs_list):
            print(f'\nOOM -> hạ batch {bs_list[i]} -> {bs_list[i+1]} và chạy lại\n', flush=True)
            resume = (not smoke) and any(OUTPUT_DIR.glob('checkpoint-*'))
            i += 1
            continue
        break
    return False

# ---------------- GIAI ĐOẠN 1: SMOKE ----------------
if AUTO_SMOKE_FIRST and not RESUME:
    clean_output()
    ok = run_phase(smoke=True)
    assert ok, 'Smoke test thất bại — xem log phía trên (dòng Traceback đầu tiên là nguyên nhân).'
    assert evaluate(5, 'smoke_predictions.json') == 0, 'Eval script lỗi — xem log phía trên.'
    print('\nSMOKE TEST THÀNH CÔNG -> bắt đầu train thật\n', flush=True)

# ---------------- GIAI ĐOẠN 2: TRAIN THẬT ----------------
if not RESUME:
    clean_output()
ok = run_phase(smoke=False, resume=RESUME)
assert ok, 'Train thất bại. Nếu bị ngắt giữa chừng: đặt RESUME=True và chạy lại từ cell này.'
print('\nTRAIN HOÀN TẤT!', flush=True)

# ---------------- GIAI ĐOẠN 3: ĐÁNH GIÁ 720 Ô M/D TRÊN GPU ----------------
print('\nBẮT ĐẦU ĐÁNH GIÁ TRÊN GPU...', flush=True)
evaluate(100, 'val_predictions.json')



In [ ]:
for d in OUTPUT_DIR.glob('checkpoint-*'):
    shutil.rmtree(d, ignore_errors=True)

print('Các file trong adapter:')
for f in sorted(ADAPTER.rglob('*')):
    if f.is_file():
        print(f'  {f.name}  ({f.stat().st_size / 1024:.0f} KB)')

zip_path = WORK / 'lora_adapter_targeted.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', str(ADAPTER))
print(f'\n{zip_path.name}: {zip_path.stat().st_size / 1024**2:.1f} MB')
print('Tải về từ tab Output (bên phải Kaggle).')
# Copy 720 eval results to /kaggle/working/
for fn in ["predictions_720.json", "summary_720.json"]:
    for f in OUTPUT_DIR.rglob(fn):
        shutil.copy(f, WORK / fn)
        print(f"Copied {fn} to /kaggle/working/")
